In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 77678)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 53207)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 51741)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 51647)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 23485)

# #### Device() ####
# device = cuda:3

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:3)
# edge_attr                (32464, 16)              Tensor (cuda:3)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:3)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

---

In [2]:
from modules.layers import AttentionSetPooling
from modules.model import BaseAutoencoder
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer
from functools import partial
import torch.nn as nn

In [ ]:
## Model

model_template = partial(
    BaseAutoencoder,
    dataset = dataset,
    method = 'node',
    mlp=False,
    variational=False,

    # layers
    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True},
)
 
model_grid = grid(
    model_template,
    embed_dim=[32,64,128,256],
    hidden_dims=[0,1,2,3,4],
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    trainer_norm_class=LogCounts,
    early_stop=True,
    stop_metric='mae',
    lr=1e-3
)

trainer_grid = grid(
    trainer_template,
)

## Experiment

expt = Experiment(
    num_trials=3,
    num_epochs=300,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

20


In [4]:
expt.run_experiment(
    'reconstr_grid1b_earlystop',
    report_metrics=['loss','rmse','mae','r2'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]